Euclidean Distance is the distance formula for 2 points in a plane in infinite dimensions.
d = sqrt((x1 - x2)^2 + (y1 - y2)^2 + (z1 - z2)^2 + ...)

Imagine a plane in two dimensions where [x1, y1] = [1, 2] and [x2, y2] = [4, 6]. The distance between these two points is:
d = sqrt((1 - 4)^2 + (2 - 6)^2)
d = sqrt((-3)^2 + (-4)^2)
d = sqrt(9 + 16)
d = sqrt(25)
d = 5

Now taking this two dimensional example and applying it to higher dimensional data where we have point X, Y = [x1, x2, x3, ..., xn], [y1, y2, y3, ..., yx]
the distance formula becomes
d = sqrt((x1 - y1)^2 + (x2 - y2)^2 + (x3 - y3)^2 + ... + (xn - yn)^2)

In [79]:
import math
import random
random.seed(67)

In [80]:
def euclidean_distance(point1, point2):
  """Calculate the Euclidean distance between two points.
  
  Time Complexity: O(D) where D is the number of dimensions
  Space Complexity: O(1)
  """
  distance = 0
  for p1, p2 in zip(point1, point2):
    distance += (p1 - p2) ** 2
  return math.sqrt(distance)

In [81]:
p1, p2 = (1, 2), (4, 6)
print(euclidean_distance(p1, p2))
# The answer is 5 as expected

5.0


In a K means algorithm the steps the algorithm takes is as follows
1. pick a random centroid (mid points) from your list of points (the K value defines how many centroids you will have)
2. calculate the distance of each point to each centroid.
3. Assign each point to the nearest centroid. The closest centroid to your point is the assignment point.
4. Move the centroid to the middle of all its assignments.
5. Reassign points to the nearest centroid again.
6. Repeat until the centroids no longer move.

In [82]:
# Step 1, pick a random centroid from the list of points
def pick_random_centroid(points, k):
  return random.sample(points, k)


In [83]:
# Step 2, calculate the distance of each point to each centroid
def find_closest_centroid(point, centroids):
  closest_centroid = 0
  closest_centroid_distance = euclidean_distance(point, centroids[0])
  for i in range(1, len(centroids)):
    distance = euclidean_distance(point, centroids[i])
    if distance < closest_centroid_distance:
      closest_centroid = i
      closest_centroid_distance = distance
  return closest_centroid

In [84]:
# Step 3, assign each point to the nearest centroid
def create_assignments(points, centroids):
  assignments = []
  for p in points:
    assignments.append(find_closest_centroid(p, centroids))
  return assignments

In [85]:
# Step 4, recalculate the centroids
def calculate_centroids(points, assignments, k, round_to = 6):
  # create group bins for each centroids
  clusters = [[] for _ in range(k)]

  # loop through points and assignments and add them to the centroid bins
  for point, assignment in zip(points, assignments):
    clusters[assignment].append(point)

  new_centroids = []

  for c in clusters:
    group_by_dimensions = list(zip(*c))
    new_centroid = ()
    for d in group_by_dimensions:
      new_centroid += (round(sum(d) / len(d), round_to),)
    new_centroids.append(new_centroid)
  return new_centroids

In [86]:
def run_algorithm(points, centroids, assignments, k):
  done = False
  iterations = 0
  while not done:
    iterations += 1
    new_centroid = calculate_centroids(points, assignments, k)
    if centroids == new_centroid:
      done = True
    else:
      centroids = new_centroid
  return centroids, iterations, assignments

In [87]:
# POINTS = [(1, 1), (9, 9), (2, 2), (8,8), (0, 1), (10, 11)]
# K = 2
# random.seed(67)
# centroids = pick_random_centroid(POINTS, K)
# assignments = create_assignments(POINTS, centroids)
# centroids, iterations, assignments = run_algorithm(POINTS, centroids, assignments)
# print("Final centroid position", centroids, "\ntook a total of", iterations, "iterations\nAssignments:",assignments)

Now testing against the Iris dataset

In [88]:
def check_accuracy(assignments, labels):
    # find unique clusters
    clusters = {}
    for assignment, label in zip(assignments, labels):
        if assignment not in clusters:
            clusters[assignment] = {}
        if label not in clusters[assignment]:
            clusters[assignment][label] = 0
        clusters[assignment][label] += 1

    # find dominant species per cluster
    mapping = {}
    for cluster_id, species_counts in clusters.items():
        dominant = max(species_counts, key=lambda s: species_counts[s])
        mapping[cluster_id] = dominant

    # count correct predictions
    correct = 0
    for assignment, label in zip(assignments, labels):
        if mapping[assignment] == label:
            correct += 1

    accuracy = correct / len(labels) * 100

    # print breakdown
    for cluster_id, species_counts in clusters.items():
        print(f"cluster {cluster_id} → mapped to '{mapping[cluster_id]}' | {species_counts}")

    print(f"\ncorrect: {correct} / {len(labels)}")
    print(f"accuracy: {accuracy:.1f}%")

    return accuracy

In [89]:
import csv
import urllib.request

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
urllib.request.urlretrieve(url, "iris.csv")

points = []
labels = [] 
with open("iris.csv") as f:
    for row in csv.reader(f):
        if row:
            points.append(tuple(float(x) for x in row[:4]))
            labels.append(row[4])

K = 3
random.seed(67)
centroids = pick_random_centroid(points, K)
assignments = create_assignments(points, centroids)
centroids, iterations, assignments = run_algorithm(points, centroids, assignments, K)
accuracy = check_accuracy(assignments, labels)

cluster 0 → mapped to 'Iris-setosa' | {'Iris-setosa': 24}
cluster 1 → mapped to 'Iris-setosa' | {'Iris-setosa': 26, 'Iris-versicolor': 7}
cluster 2 → mapped to 'Iris-virginica' | {'Iris-versicolor': 43, 'Iris-virginica': 50}

correct: 100 / 150
accuracy: 66.7%
